# Spike: conservação contrastiva

**Objetivo:** testar, com dados 100% sintéticos, uma etapa intermediária entre conservação do alvo e desenho de assays.

A pergunta do spike é:

> Quais janelas são muito conservadas dentro do conjunto-alvo e, ao mesmo tempo, pouco semelhantes ao painel de não alvos?

Fluxo conceitual:

`TARGET variants → conservação no alvo → contraste com NON-TARGETS → janelas discriminantes → (futuro) Primer3 → especificidade final do assay`

Este notebook **não desenha primers**, não usa organismos reais e não propõe thresholds biológicos. Os limites abaixo são apenas ilustrativos para avaliar a arquitetura.


In [ ]:
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 20260903
LENGTH = 1200
TARGET_COUNT = 8
NON_TARGET_COUNT = 5
WINDOW = 80
STEP = 10

# Apenas para este fixture sintético. Não são defaults científicos.
MIN_TARGET_CONSERVATION = 0.985
MAX_NON_TARGET_SIMILARITY = 0.85

rng = random.Random(SEED)
BASES = "ACGT"

def random_sequence(length):
    return "".join(rng.choice(BASES) for _ in range(length))

def mutate(sequence, probability_by_position):
    out = []
    for i, base in enumerate(sequence):
        p = probability_by_position(i)
        if rng.random() < p:
            out.append(rng.choice([b for b in BASES if b != base]))
        else:
            out.append(base)
    return "".join(out)

reference = random_sequence(LENGTH)
print(f"Synthetic reference length: {len(reference)}")


## 1. Construir o cenário sintético

O fixture contém duas regiões deliberadas:

- **shared-conserved (250–449):** conservada no alvo **e também** nos não alvos. Deve ser rejeitada pelo contraste.
- **discriminant (700–899):** conservada no alvo, mas divergente nos não alvos. Deve aparecer entre as melhores candidatas.

Fora dessas regiões há variabilidade suficiente para o notebook não produzir um resultado trivial.


In [ ]:
def target_mutation_probability(i):
    if 250 <= i < 450 or 700 <= i < 900:
        return 0.005
    return 0.05

def nontarget_mutation_probability(i):
    if 250 <= i < 450:
        return 0.02   # região compartilhada: ainda muito parecida
    if 700 <= i < 900:
        return 0.35   # região discriminante: propositalmente divergente
    return 0.20

targets = {
    f"target-{i+1}": mutate(reference, target_mutation_probability)
    for i in range(TARGET_COUNT)
}
non_targets = {
    f"non-target-{i+1}": mutate(reference, nontarget_mutation_probability)
    for i in range(NON_TARGET_COUNT)
}

print(f"TARGET sequences: {len(targets)}")
print(f"NON-TARGET sequences: {len(non_targets)}")
print("Expected synthetic zones: shared-conserved=250-449, discriminant=700-899")


## 2. Medir conservação no alvo e similaridade nos não alvos

Para o spike usamos sequências sintéticas do mesmo comprimento e coordenadas já correspondentes.

Para cada janela calculamos:

- **target_conservation:** média da frequência da base majoritária por posição entre as sequências-alvo;
- **non_target_similarity:** maior identidade observada entre o consenso do alvo e qualquer não alvo;
- **contrast_score:** `target_conservation - non_target_similarity`.

A hipótese de arquitetura é que uma etapa real do Geison poderia produzir um artefato equivalente antes do `primer_design`.


In [ ]:
def consensus(sequences):
    seqs = list(sequences)
    return "".join(
        Counter(seq[i] for seq in seqs).most_common(1)[0][0]
        for i in range(len(seqs[0]))
    )

def mean_position_conservation(sequences, start, end):
    seqs = list(sequences)
    n = len(seqs)
    values = []
    for i in range(start, end):
        top = Counter(seq[i] for seq in seqs).most_common(1)[0][1]
        values.append(top / n)
    return float(np.mean(values))

def identity(a, b):
    return sum(x == y for x, y in zip(a, b)) / len(a)

target_consensus = consensus(targets.values())
rows = []
for start in range(0, LENGTH - WINDOW + 1, STEP):
    end = start + WINDOW
    target_conservation = mean_position_conservation(targets.values(), start, end)
    consensus_window = target_consensus[start:end]
    similarities = [
        identity(consensus_window, seq[start:end])
        for seq in non_targets.values()
    ]
    non_target_similarity = max(similarities)
    rows.append({
        "start": start + 1,
        "end": end,
        "midpoint": (start + 1 + end) / 2,
        "target_conservation": target_conservation,
        "non_target_similarity": non_target_similarity,
        "contrast_score": target_conservation - non_target_similarity,
    })

windows = pd.DataFrame(rows)
windows["candidate"] = (
    (windows.target_conservation >= MIN_TARGET_CONSERVATION)
    & (windows.non_target_similarity <= MAX_NON_TARGET_SIMILARITY)
)

candidates = windows[windows.candidate].sort_values(
    ["contrast_score", "target_conservation"], ascending=False
).reset_index(drop=True)

print(f"Windows evaluated: {len(windows)}")
print(f"Candidate windows: {len(candidates)}")
assert len(candidates) > 0, "Synthetic spike should produce at least one discriminant candidate"
assert candidates.iloc[0].midpoint >= 650 and candidates.iloc[0].midpoint <= 950, (
    "Best synthetic candidate should land near the deliberately discriminant zone"
)


## 3. Visual 1: o quadrante que interessa

Quanto mais **alto** no eixo Y, mais conservada a janela é dentro do alvo. Quanto mais **à esquerda** no eixo X, menos semelhante ela é aos não alvos.

O quadrante superior esquerdo representa a ideia que queremos testar: **alta inclusividade potencial + alta exclusividade potencial**.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(
    windows.non_target_similarity,
    windows.target_conservation,
    alpha=0.45,
    label="all windows",
)
ax.scatter(
    candidates.non_target_similarity,
    candidates.target_conservation,
    s=55,
    label="candidate windows",
)
ax.axhline(MIN_TARGET_CONSERVATION, linestyle="--", linewidth=1)
ax.axvline(MAX_NON_TARGET_SIMILARITY, linestyle="--", linewidth=1)
ax.set_xlabel("Maximum similarity to any non-target")
ax.set_ylabel("Mean conservation within target set")
ax.set_title("Contrastive conservation: target vs non-targets")
ax.legend()
plt.show()


## 4. Visual 2: onde essas janelas ficam ao longo da referência sintética

Esse gráfico deixa explícita a diferença entre:

- uma região apenas **conservada**;
- uma região **conservada e discriminante**.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(windows.midpoint, windows.target_conservation, label="target conservation")
ax.plot(windows.midpoint, windows.non_target_similarity, label="max non-target similarity")
for _, row in candidates.iterrows():
    ax.axvspan(row.start, row.end, alpha=0.08)
ax.axvspan(250, 450, alpha=0.08, label="shared-conserved zone")
ax.axvspan(700, 900, alpha=0.08, label="discriminant zone")
ax.set_ylim(0, 1.02)
ax.set_xlabel("Synthetic reference coordinate")
ax.set_ylabel("Score")
ax.set_title("Target conservation and non-target similarity along the synthetic reference")
ax.legend(loc="lower left", ncol=2)
plt.show()


## 5. Ranking experimental de janelas

O `contrast_score` é propositalmente simples. Ele serve apenas para mostrar qual tipo de artefato uma futura etapa do Geison poderia entregar ao `primer_design`.


In [ ]:
display_cols = [
    "start", "end", "target_conservation",
    "non_target_similarity", "contrast_score"
]

display(candidates[display_cols].head(12).round(4))

outdir = Path("/content/contrastive_conservation_spike")
outdir.mkdir(parents=True, exist_ok=True)
windows.to_csv(outdir / "all_windows.csv", index=False)
candidates.to_csv(outdir / "candidate_windows.csv", index=False)
print(f"Saved: {outdir / 'candidate_windows.csv'}")


## Leitura do spike

Se o notebook se comportar como esperado, ele sustenta esta arquitetura para o Geison:

1. **Conservation** continua respondendo “o que é estável dentro do alvo?”.
2. Uma nova etapa, provisoriamente chamada **Contrastive Conservation**, responde “o que é estável no alvo e divergente nos não alvos?”.
3. **Primer design** recebe regiões pré-filtradas/rankeadas por esse contraste.
4. **Specificity** continua depois do Primer3, agora no nível dos oligos/assay, como verificação final independente.

Isso evita usar a especificidade final para resolver uma pergunta que existe **antes** do desenho dos oligos.

### O que este spike não resolve

- alinhamento entre espécies/organismos;
- escolha automática do painel de não alvos;
- thresholds científicos;
- ponderação diferente entre não alvos críticos e contextuais;
- desenho ou validação de primers/sondas.

Esses pontos só devem virar requisitos de produção depois que a separação conceitual acima for aprovada.
